## Chuẩn bị dữ liệu

In [104]:
# Import library
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
# from sklearn.metrics import plot_confusion_matrix
import category_encoders as ce
from imblearn.over_sampling import SMOTE, SVMSMOTE

In [105]:
customers = pd.read_csv("../data/2_clean/customers.csv")
orders = pd.read_csv("../data/2_clean/orders.csv")
order_items = pd.read_csv("../data/2_clean/order_items.csv")
payments = pd.read_csv("../data/2_clean/payments.csv")
products = pd.read_csv("../data/2_clean/products.csv")
reviews = pd.read_csv("../data/2_clean/reviews.csv")
geolocation = pd.read_csv("../data/2_clean/geolocation.csv")
sellers = pd.read_csv("../data/2_clean/sellers.csv")

## Chia tập train, test

In [107]:
# Chuyển cột ngày tháng sang dạng datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# 3. Sắp xếp dữ liệu theo thứ tự thời gian tăng dần
orders = orders.sort_values(by='order_purchase_timestamp')

# Kiểm tra khoảng thời gian của dữ liệu
print(f"Ngày bắt đầu: {orders['order_purchase_timestamp'].min()}")
print(f"Ngày kết thúc: {orders['order_purchase_timestamp'].max()}")

Ngày bắt đầu: 2016-09-04 21:15:19
Ngày kết thúc: 2018-10-17 17:30:18


#### Chọn 80 train, 20 test
Tổng 25 tháng => Lấy mốc `01/04/2018` (6 tháng)

In [108]:
def split_orders(orders, input_date):
    # Chọn ngày cắt
    split_date = pd.Timestamp(input_date)

    # Tạo mask lọc
    train_mask = orders['order_purchase_timestamp'] < split_date
    test_mask = orders['order_purchase_timestamp'] >= split_date

    # Chia dữ liệu
    orders_train_df = orders.loc[train_mask]
    orders_test_df = orders.loc[test_mask]

    # Kiểm tra tỷ lệ
    n_total = len(orders)
    n_train = len(orders_train_df)
    n_test = len(orders_test_df)

    print(f"Số lượng dòng Train: {n_train} ({n_train/n_total:.1%})")
    print(f"Số lượng dòng Test: {n_test} ({n_test/n_total:.1%})")

    return orders_train_df, orders_test_df

orders_train_df, orders_test_df = split_orders(orders, '2018-04-01')

Số lượng dòng Train: 66638 (67.0%)
Số lượng dòng Test: 32803 (33.0%)


#### 
Nới mốc chia sang `01/06/2018`

In [109]:
orders_train_df, orders_test_df = split_orders(orders, '2018-06-01')

Số lượng dòng Train: 80450 (80.9%)
Số lượng dòng Test: 18991 (19.1%)


## Feature engineering

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Utility: safe qcut (robust)
# -----------------------------
def safe_qcut(series, q=5, labels=None):
    """Robust quantile binning."""
    if labels is None:
        labels = list(range(1, q+1))
    s = series.fillna(-999999999).copy()
    try:
        ranked = s.rank(method='first', pct=True)
        bins = np.linspace(0, 1, q+1)
        return pd.cut(ranked, bins=bins, labels=labels, include_lowest=True)
    except Exception:
        return pd.Series(pd.cut(s.rank(method='first'), bins=q, labels=labels, 
                               include_lowest=True), index=series.index)

# -----------------------------
# 2) Calculate average review score
# -----------------------------
def calculate_avg_review_score(customers, orders_sub, reviews):
    """Tính điểm review trung bình cho mỗi khách hàng"""
    if reviews is None or reviews.empty:
        return pd.DataFrame({
            'customer_unique_id': pd.Series(dtype='str'),
            'avg_review_score': pd.Series(dtype='float')
        })
    
    # Merge đơn giản
    df = customers[['customer_id', 'customer_unique_id']].merge(
        orders_sub[['order_id', 'customer_id']], on='customer_id', how='inner'  # Chỉ lấy khách hàng có đơn hàng
    ).merge(
        reviews[['order_id', 'review_score']], on='order_id', how='left'
    )
    
    # Tính trung bình
    result = df.groupby('customer_unique_id', as_index=False)['review_score'].mean()
    result.rename(columns={'review_score': 'avg_review_score'}, inplace=True)
    result['avg_review_score'] = result['avg_review_score'].fillna(0).round(2)
    
    return result

# -----------------------------
# 3) Calculate core metrics
# -----------------------------
def calculate_core_metrics(customers, orders_sub, payments, last_date):
    """
    Tính các metrics cốt lõi: RFM và delivery metrics
    """
    # Chỉ merge với khách hàng có đơn hàng trong orders_sub
    df = customers[['customer_id', 'customer_unique_id', 'customer_state']].merge(
        orders_sub, on='customer_id', how='inner'  # INNER JOIN thay vì LEFT JOIN
    ).copy()
    
    # Kiểm tra nếu không có dữ liệu
    if df.empty:
        return pd.DataFrame(columns=[
            'customer_unique_id', 'first_purchase', 'last_purchase', 
            'total_orders', 'customer_state', 'monetary',
            'delivered_orders', 'num_late_deliveries', 'avg_delivery_days',
            'recency_days', 'customer_age_days', 'late_delivery_rate',
            'avg_order_value', 'orders_per_month'
        ])
    
    # Chuyển đổi datetime
    for col in ['order_purchase_timestamp', 'order_estimated_delivery_date', 
                'order_delivered_customer_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Tính payment value nếu có payments data
    if payments is not None and not payments.empty:
        # Lọc payments chỉ cho các order trong orders_sub
        order_ids = orders_sub['order_id'].unique()
        payments_sub = payments[payments['order_id'].isin(order_ids)].copy()
        payments_agg = payments_sub.groupby('order_id')['payment_value'].sum().reset_index()
        df = df.merge(payments_agg, on='order_id', how='left')
        df['payment_value'] = df['payment_value'].fillna(0)
    else:
        df['payment_value'] = 0
    
    # Aggregate cơ bản
    agg_all = df.groupby('customer_unique_id', as_index=False).agg(
        first_purchase=('order_purchase_timestamp', 'min'),
        last_purchase=('order_purchase_timestamp', 'max'),
        total_orders=('order_id', 'nunique'),
        customer_state=('customer_state', 'first'),
        monetary=('payment_value', 'sum')
    )
    
    # Metrics cho delivered orders
    df_delivered = df[df['order_status'] == 'delivered'].copy()
    if not df_delivered.empty:
        df_delivered['delivery_days'] = (
            pd.to_datetime(df_delivered['order_delivered_customer_date'], errors='coerce') -
            pd.to_datetime(df_delivered['order_purchase_timestamp'], errors='coerce')
        ).dt.days
        
        df_delivered['late_delivery'] = (
            pd.to_datetime(df_delivered['order_delivered_customer_date'], errors='coerce') >
            pd.to_datetime(df_delivered['order_estimated_delivery_date'], errors='coerce')
        ).astype(int)
        
        agg_delivered = df_delivered.groupby('customer_unique_id', as_index=False).agg(
            delivered_orders=('order_id', 'nunique'),
            num_late_deliveries=('late_delivery', 'sum'),
            avg_delivery_days=('delivery_days', 'mean')
        )
    else:
        agg_delivered = pd.DataFrame({
            'customer_unique_id': pd.Series(dtype='str'),
            'delivered_orders': pd.Series(dtype='int'),
            'num_late_deliveries': pd.Series(dtype='int'),
            'avg_delivery_days': pd.Series(dtype='float')
        })
    
    # Merge và tính toán thêm metrics
    result = agg_all.merge(agg_delivered, on='customer_unique_id', how='left')
    
    # Fill NaN values
    for col in ['delivered_orders', 'num_late_deliveries', 'avg_delivery_days']:
        result[col] = result[col].fillna(0)
    
    # Tính toán RFM metrics
    last_date = pd.to_datetime(last_date)
    result['recency_days'] = (last_date - result['last_purchase']).dt.days
    result['recency_days'] = result['recency_days'].fillna(99999).astype(int)
    
    result['customer_age_days'] = (last_date - result['first_purchase']).dt.days
    result['customer_age_days'] = result['customer_age_days'].fillna(0).astype(int)
    
    result['late_delivery_rate'] = (result['num_late_deliveries'] / 
                                   result['delivered_orders'].replace(0, 1)).round(2)
    
    result['avg_order_value'] = (result['monetary'] / 
                                result['delivered_orders'].replace(0, 1)).round(2)
    
    result['orders_per_month'] = np.where(
        result['customer_age_days'] > 30,
        (result['total_orders'] / (result['customer_age_days'] / 30)).round(2),
        0
    )
    
    return result

# -----------------------------
# 4) Calculate RFM scores
# -----------------------------
def calculate_rfm_scores(df):
    """Tính RFM scores"""
    # Recency: lower is better (5 = most recent)
    df['r_score'] = safe_qcut(df['recency_days'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
    
    # Frequency: higher is better (5 = most frequent)
    df['f_score'] = safe_qcut(df['delivered_orders'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    
    # Monetary: higher is better (5 = highest spending)
    df['m_score'] = safe_qcut(df['monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    
    # Combined RFM score
    df['rfm_score'] = df['r_score'].astype(str) + df['f_score'].astype(str) + df['m_score'].astype(str)
    
    return df

# -----------------------------
# 5) Select final features
# -----------------------------
def select_final_features(df):
    """Chọn final features cho model"""
    core_features = [
        "customer_unique_id",
        # RFM metrics
        "recency_days", "delivered_orders", "monetary",
        "r_score", "f_score", "m_score", "rfm_score",
        # Behavioral metrics
        "total_orders", "orders_per_month", "avg_order_value",
        # Delivery metrics
        "num_late_deliveries", "late_delivery_rate", "avg_delivery_days",
        # Review score
        "avg_review_score",
        # Demographic
        "customer_state"
    ]
    
    # Tạo các features còn thiếu nếu cần
    for feature in core_features:
        if feature not in df.columns:
            if feature == "customer_unique_id":
                continue
            elif feature in ["customer_state"]:
                df[feature] = ""
            else:
                df[feature] = 0
    
    return df[core_features]

# -----------------------------
# 6) Main pipeline
# -----------------------------
def generate_features(customers, orders_sub, payments, reviews, last_date):
    """
    Pipeline chính để tạo features
    """
    # 1. Tính core metrics
    features = calculate_core_metrics(customers, orders_sub, payments, last_date)
    
    # Kiểm tra nếu không có features
    if features.empty:
        # Tạo dataframe rỗng với đúng cấu trúc
        empty_features = pd.DataFrame(columns=[
            'customer_unique_id', 'recency_days', 'delivered_orders', 'monetary',
            'r_score', 'f_score', 'm_score', 'rfm_score', 'total_orders',
            'orders_per_month', 'avg_order_value', 'num_late_deliveries',
            'late_delivery_rate', 'avg_delivery_days', 'avg_review_score',
            'customer_state'
        ])
        return empty_features
    
    # 2. Tính avg_review_score
    if reviews is not None and not reviews.empty:
        # Lọc reviews chỉ cho các order trong orders_sub
        order_ids = orders_sub['order_id'].unique()
        reviews_sub = reviews[reviews['order_id'].isin(order_ids)].copy()
        avg_review_df = calculate_avg_review_score(customers, orders_sub, reviews_sub)
        features = features.merge(avg_review_df, on='customer_unique_id', how='left')
    else:
        features['avg_review_score'] = 0.0
    
    # Fill NaN cho avg_review_score
    features['avg_review_score'] = features['avg_review_score'].fillna(0.0)
    
    # 3. Tính RFM scores
    features = calculate_rfm_scores(features)
    
    # 4. Chọn final features
    features = select_final_features(features)
    
    return features

# -----------------------------
# 7) Create labels
# -----------------------------
def create_labels(customers, orders, perf_start, perf_end):
    """Tạo label is_repurchase"""
    perf_start = pd.to_datetime(perf_start)
    perf_end = pd.to_datetime(perf_end)
    
    # Lấy orders trong performance window
    perf_orders = orders[
        (pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce') >= perf_start) &
        (pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce') <= perf_end)
    ].copy()
    
    # Tạo mapping từ customer_unique_id sang customer_id
    customer_mapping = customers[['customer_unique_id', 'customer_id']].drop_duplicates()
    mapping_dict = dict(zip(customer_mapping['customer_unique_id'], 
                           customer_mapping['customer_id']))
    
    # Lấy danh sách customer_id có repurchase
    repurchase_customers = set(perf_orders['customer_id'].unique())
    
    # Tạo labels
    labels = pd.DataFrame({
        'customer_unique_id': customers['customer_unique_id'].unique()
    })
    
    labels['is_repurchase'] = labels['customer_unique_id'].map(
        lambda x: 1 if mapping_dict.get(x) in repurchase_customers else 0
    )
    
    return labels

# =============================
# MAIN EXECUTION
# =============================

# Giả sử bạn đã có các dataframe:
# customers, orders, payments, reviews

# 1. Chuyển đổi datetime cho orders
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'], errors='coerce'
)

# 2. Split data
split_date = '2018-06-01'
train_orders, test_orders = split_orders(orders, split_date)

# 3. Tạo features
last_date_train = pd.to_datetime(split_date)
last_date_test = orders['order_purchase_timestamp'].max()

# Thêm reviews vào hàm generate_features
train_features = generate_features(customers, train_orders, payments, reviews, last_date_train)
test_features = generate_features(customers, test_orders, payments, reviews, last_date_test)

print(f"\nTrain features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")

# 4. Tạo labels cho train
perf_start = pd.to_datetime(split_date)
perf_end = perf_start + pd.Timedelta(days=90)

train_labels = create_labels(customers, orders, perf_start, perf_end)

# 5. Merge labels với features (chỉ lấy khách hàng có trong train_features)
train_data = train_features.merge(train_labels, on='customer_unique_id', how='inner')
train_data['is_repurchase'] = train_data['is_repurchase'].fillna(0).astype(int)

print(f"\nClass distribution:")
print(train_data['is_repurchase'].value_counts(normalize=True))

# 6. Kiểm tra features
print(f"\nFeatures used: {list(train_features.columns)}")
print(f"\nSố lượng khách hàng có đơn hàng trong train: {len(train_features)}")
print(f"Số lượng khách hàng có đơn hàng trong test: {len(test_features)}")

Số lượng dòng Train: 80450 (80.9%)
Số lượng dòng Test : 18991 (19.1%)

Train features shape: (77808, 16)
Test features shape: (18728, 16)

Class distribution:
is_repurchase
0    0.997031
1    0.002969
Name: proportion, dtype: float64

Features used: ['customer_unique_id', 'recency_days', 'delivered_orders', 'monetary', 'r_score', 'f_score', 'm_score', 'rfm_score', 'total_orders', 'orders_per_month', 'avg_order_value', 'num_late_deliveries', 'late_delivery_rate', 'avg_delivery_days', 'avg_review_score', 'customer_state']

Số lượng khách hàng có đơn hàng trong train: 77808
Số lượng khách hàng có đơn hàng trong test: 18728


In [120]:
train_data

,customer_unique_id,recency_days,delivered_orders,monetary,r_score,f_score,m_score,rfm_score,total_orders,orders_per_month,avg_order_value,num_late_deliveries,late_delivery_rate,avg_delivery_days,avg_review_score,customer_state,is_repurchase
0,0000366f3b9a7992bf8c76cfdf3221e2,21,1.0,141.90,5,1,4,514,1,0.00,141.90,0.0,0.0,6.0,5.0,SP,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,24,1.0,27.19,5,1,1,511,1,0.00,27.19,0.0,0.0,3.0,4.0,SP,0
2,0000f46a3911fa3c0805444483337064,447,1.0,86.22,1,1,2,112,1,0.07,86.22,0.0,0.0,25.0,3.0,SC,0
3,0000f6ccb0745a6a4b88665a16c9f078,231,1.0,43.62,2,1,1,211,1,0.13,43.62,0.0,0.0,20.0,4.0,PA,0
4,0004aac84e0df4da2b147fca70cf8255,198,1.0,196.89,3,1,4,314,1,0.15,196.89,0.0,0.0,13.0,5.0,SP,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77803,fffcf5a5ff07b0908bd4e2dbc735a684,357,1.0,2067.42,1,5,5,155,1,0.08,2067.42,0.0,0.0,27.0,5.0,PE,0
77804,fffea47cd6d3cc0a88bd621562a9d061,172,1.0,84.58,3,5,2,352,1,0.17,84.58,0.0,0.0,30.0,4.0,BA,0
77805,ffff371b4d645b6ecea244b27531430a,478,1.0,112.46,1,5,3,153,1,0.06,112.46,0.0,0.0,14.0,5.0,MT,0
77806,ffff5962728ec6157033ef9805bacc48,29,1.0,133.69,5,5,4,554,1,0.00,133.69,0.0,0.0,11.0,5.0,ES,0


In [121]:
train_data["is_repurchase"].value_counts()

is_repurchase
0    77577
1      231
Name: count, dtype: int64

In [ ]:
train_data.to_csv("../data/3_model/train.csv", index=False)
test_features.to_csv("../data/3_model/test.csv", index=False)

## Churn

In [116]:
df = pd.read_csv("../data/3_model/train.csv")

In [117]:
df

,customer_unique_id,recency_days,delivered_orders,monetary,r_score,f_score,m_score,rfm_score,total_orders,orders_per_month,avg_order_value,num_late_deliveries,late_delivery_rate,avg_delivery_days,avg_review_score,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,21,1.0,141.90,5,2,4,524,1,0.00,141.90,0.0,0.0,6.0,5.0,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,24,1.0,27.19,5,2,2,522,1,0.00,27.19,0.0,0.0,3.0,4.0,SP
2,0000f46a3911fa3c0805444483337064,447,1.0,86.22,2,2,3,223,1,0.07,86.22,0.0,0.0,25.0,3.0,SC
3,0000f6ccb0745a6a4b88665a16c9f078,231,1.0,43.62,3,2,2,322,1,0.13,43.62,0.0,0.0,20.0,4.0,PA
4,0004aac84e0df4da2b147fca70cf8255,198,1.0,196.89,3,2,5,325,1,0.15,196.89,0.0,0.0,13.0,5.0,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,357,1.0,2067.42,2,5,5,255,1,0.08,2067.42,0.0,0.0,27.0,5.0,PE
96092,fffea47cd6d3cc0a88bd621562a9d061,172,1.0,84.58,4,5,3,453,1,0.17,84.58,0.0,0.0,30.0,4.0,BA
96093,ffff371b4d645b6ecea244b27531430a,478,1.0,112.46,2,5,4,254,1,0.06,112.46,0.0,0.0,14.0,5.0,MT
96094,ffff5962728ec6157033ef9805bacc48,29,1.0,133.69,5,5,4,554,1,0.00,133.69,0.0,0.0,11.0,5.0,ES
